In [ ]:
# from openai import OpenAI

# # 初始化客户端
# client = OpenAI(
#     api_key="sk-qGPvtEBthIIULdhf61A9D2AfC09244B0809e2a2547A69dE1",
#     base_url="https://ds-api.yovole.com/v1"
# )

# # 获取模型列表
# models = client.models.list()
# print("可用模型：")
# for model in models.data:
#     print(model.id)

可用模型：
bge-m3
claude-haiku-4-5-20251001
deepseek-ai/DeepSeek-R1-0528-Qwen3-8B
DeepSeek-OCR
DeepSeek-R1
DeepSeek-R1-0528-Qwen3-8B
DeepSeek-R1-0528-Qwen3-8B-Instruct
DeepSeek-V3
DeepSeek-V3.1
DeepSeek-V3.2
eduChat-32B
glm-4.6
gpt-oss-120b
gpt-oss-20b
kimi-k2-instruct
kolors
nim.deepseek-r1-distill-llama-8b
nim.llama-3.3-70b-instruct
nim.qwen-2.5-7b-instruct
Qwen-Image
Qwen2.5-32B-Instruct
Qwen2.5-72B-Instruct
Qwen2.5-7B-Instruct
Qwen2.5-VL-32B-Instruct
Qwen2.5-VL-72B-Instruct
Qwen3-14B
Qwen3-30B-A3B
Qwen3-32B
qwen3-coder-plus
Qwen3-Embedding-8B
Qwen3-Next-80B-A3B-Instruct
Qwen3-VL-30B-A3B-Instruct
Qwen3Guard-Gen-8B
QwQ-32B


In [ ]:
# # 与模型对话
# completion = client.chat.completions.create(
#     model="gpt-oss-120b",
#     messages=[
#         {"role": "user", "content": "你好，你是什么模型？你的MAX_TOKENS是多少？"}
#     ]
# )

# # 打印模型的回复
# print("\n模型回复：")
# print(completion.choices[0].message.content)


模型回复：
你好！我是基于 **GPT‑4** 架构的 ChatGPT 模型（也有 32k 版本可供使用）。  

- **上下文窗口（Context window）**  
  - 标准版 GPT‑4：约 **8192 tokens**（≈ 12 KB 文本）。  
  - 大容量版 GPT‑4‑32k：约 **32768 tokens**（≈ 48 KB 文本）。  

- **单次生成的最大长度（MAX_TOKENS）**  
  - 在实际对话中，我的回复通常会被限制在 **大约 2048–4096 tokens** 之间，这取决于当前会话已占用的上下文以及后台设置的安全阈值。  
  - 如果你使用 API 调用，可以自行指定 `max_tokens` 参数，最高可以接近剩余的上下文窗口大小（例如在标准 8k 模型下最多约 6000 tokens，具体取决于已有的输入占用了多少）。

简而言之，我是 GPT‑4 模型，标准上下文上限约 8 k tokens，生成时通常会限制在几千个 token 以内。希望这能解答你的疑问！如果还有其他问题，随时告诉我。


# 库引入与关键函数

In [1]:
import pandas as pd
import openai
from openai import OpenAI
import numpy as np
import time
import os
import glob
import tiktoken
import nltk
import re
import tqdm
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

# --- 初始化与配置 ---
MODEL = 'DeepSeek-V3.2'  # 或者使用 'gpt-3.5-turbo' 等
MAX_TOKENS = 8000
WAIT_TIME = 0.8

client = OpenAI(
    api_key="sk-qGPvtEBthIIULdhf61A9D2AfC09244B0809e2a2547A69dE1",
    base_url="https://ds-api.yovole.com/v1"
)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
# --- 数据加载与预处理函数 ---
def is_latin_alphabet(text):
    if not text: return False
    latin_characters = sum(1 for char in text if ord(char) <= 0x007F)
    return (latin_characters / len(text)) >= 0.9

def split_text_into_chunks(text, max_tokens=6000):
    #encoding = tiktoken.encoding_for_model(MODEL)  #@@
    encoding = tiktoken.get_encoding("cl100k_base")
    if len(encoding.encode(text)) < max_tokens:
        return [text]
    
    sentences = sent_tokenize(text)
    #sentences = re.split(r'(?<=[。！？.!?;])\s*', text)##@@@@@
    chunks, current_chunk, current_length = [], [], 0
    
    for sentence in sentences:
        #if not sentence.strip(): continue##@@@@@
        sent_len = len(encoding.encode(sentence))
        if current_length + sent_len > max_tokens:
            chunks.append(" ".join(current_chunk))
            current_chunk, current_length = [sentence], sent_len
        else:
            current_chunk.append(sentence)
            current_length += sent_len
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

# --- API 调用核心函数 ---
def call_llm(text, instruction):
    tries = 0
    while tries < 5:
        try:
            response = client.chat.completions.create(
                model=MODEL,
                temperature=0.2, # 提取任务建议温度设为0.2，可更改看效果@@
                messages=[
                    {"role": "system", "content": instruction},
                    {"role": "user", "content": f"文本内容：'{text}'"}
                ],
                timeout=120.0  #回答时长上限，超过自动重来一次@@
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"\n[Warning] API调用出错: {e}。正在进行第 {tries+1} 次重试...")
            time.sleep(10)
            tries += 1
    return "[]"


def parse_list_result(result_str):
    """解析LLM返回的列表字符串"""
    try:
        # 清洗掉可能存在的markdown代码块标记
        clean_str = result_str.strip().replace("```python", "").replace("```", "").strip()
        return eval(clean_str) # 将字符串转为list
    except:
        return []

# 一、教育文献分析

### 1. Prompt

In [3]:
#任务一：提炼关键句；任务二：根据关键句回答问题
#任务一和任务二的prompt
EXTRACTION_INSTRUCTION = """
你的任务是从提供的文本块中，找出与以下话题相关的论述：
1. 科特迪瓦公立小学与私立小学的收费标准、各方面的差异。
2. 科特迪瓦的城乡差异（经济、教育等方面）。

请直接输出找到的论述原文（保持原文语言，不要翻译），严禁凭空臆造。
输出格式必须是一个 Python 列表格式，例如：
["原文论述1", "原文论述2", "原文论述3"]
如果没找到相关论述，请输出：[]。不要输出任何多余的解释文字。
"""

EXPERT_PROMPT = """
你是一个教育、经济相关领域的资深专家。请深入分析以下来自科特迪瓦相关文件的论述，回答问题：
“推断科特迪瓦城市/乡村、公立/私立小学学生家庭的平均经济水平及其差异”。

要求：
1. 确保真实性与逻辑性。
2. 回答完后，必须明确列出你的各个观点分别是基于哪些文件的哪些论述得出的。
"""

### 2. 任务一：提炼各文件关键句

In [17]:
# 加载文件
folder_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Unstructured Data for LLM\txt about education'

txt_files = glob.glob(os.path.join(folder_path, '*.txt'))
data = []
for file_path in txt_files:
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
        data.append({'filename': os.path.basename(file_path), 'text': content})

df = pd.DataFrame(data)
# 过滤非拉丁字符
# df = df[df['text'].apply(is_latin_alphabet)]

# 分块
df['text_chunks'] = df['text'].apply(lambda x: split_text_into_chunks(x))
df['relevant_description'] = [[] for _ in range(len(df))]


# --- 优化后的执行主流程 ---
print(f"开始处理，共计 {len(df)} 个文件...")
for index, row in df.iterrows():
    all_findings = []
    chunks = row['text_chunks']
    print(f"\n正在处理文件 [{index+1}/{len(df)}]: {row['filename']} (共 {len(chunks)} 个分块)")
    
    # 为分块处理增加内部进度条
    for i, chunk in enumerate(chunks):
        #print(chunk)#######@@
        # 实时打印进度
        print(f"  -> 正在提取第 {i+1}/{len(chunks)} 块...", end='\r') 
        
        raw_result = call_llm(chunk, EXTRACTION_INSTRUCTION)
        print(f"LLM原文返回: {raw_result}")############@@@@
        findings = parse_list_result(raw_result)
        all_findings.extend(findings)
        time.sleep(WAIT_TIME)
    
    df.at[index, 'relevant_description'] = list(set(all_findings))
    print(f"\n  Done! 该文件提取到 {len(all_findings)} 条论述。")

开始处理，共计 28 个文件...

正在处理文件 [1/28]: 110791-P156307-CoteD-Ivoire-reportcard-long-v3-print.txt (共 1 个分块)
LLM原文返回: ["Notable differences emerge between public and private schools and between urban and rural areas."]

  Done! 该文件提取到 1 条论述。

正在处理文件 [2/28]: 247040fre.txt (共 47 个分块)
LLM原文返回: []1/47 块...
LLM原文返回: []2/47 块...
LLM原文返回: []3/47 块...
LLM原文返回: []4/47 块...
LLM原文返回: ["Le phénomène de non-scolarisation et d’abandons précoces affectent le plus les couches les plus pauvres de la population, les jeunes en milieu rural et de façon prépondérante les jeunes des régions du Nord et du Sud-Ouest.", "Par ailleurs, 72 % des EHSS âgés de 6 a 11 ans résident en milieu rural, alors qu’ils ne sont que 28 % a résider en milieu urbain, suggérant que le défi de la scolarisation primaire universelle et plus tard de l’achèvement universel d’un enseignement de base de dix ans passe nécessairement par la capacité de la Côte d’lvoire à pouvoir offrir une scolarité primaire complète aux enfants ruraux.", "En 20

### 3. 任务二：根据关键句回答问题

In [18]:
# 汇总所有非空的论述
summary_context = ""
for _, row in df.iterrows():
    if row['relevant_description']:
        summary_context += f"--- 文件: {row['filename']} ---\n"
        summary_context += "\n".join(row['relevant_description']) + "\n\n"


print("\n正在生成专家分析报告...")
final_analysis = call_llm(summary_context, EXPERT_PROMPT)
print("\n--- 任务二：专家分析结果 ---")
print(final_analysis)


正在生成专家分析报告...

--- 任务二：专家分析结果 ---
根据提供的文件内容，我对科特迪瓦城市/乡村、公立/私立小学学生家庭的平均经济水平及其差异进行如下深入分析：

### **核心结论**
科特迪瓦小学学生家庭的经济水平存在显著的**城乡差异**和**公私立学校差异**。总体而言，**城市家庭、尤其是选择私立学校的城市家庭，经济水平显著高于农村家庭和公立学校学生家庭**。农村地区和公立学校系统集中了该国最贫困的人口，而私立教育（主要分布在城市）则更多地服务于经济条件较好的家庭。

### **详细分析**

#### **1. 城乡差异：经济水平鸿沟显著**
*   **农村家庭普遍更贫困**：多个文件明确指出，贫困在乡村地区更为集中和严重。例如，一份报告指出“贫困在乡村地区（56.8%）比在城市地区（35.9%）更为突出”（文件247040fre）。另一份2024年的报告也确认“贫困在农村地区尤为明显，货币贫困率为54.5%，显著高于全国平均水平”（文件Annual-Report-Cote D‘Ivoire-2024）。这表明农村家庭的平均经济水平远低于城市家庭。
*   **教育支出差距巨大**：家庭经济水平的差异直接反映在教育投入上。城市家庭为每个小学生的教育支出是农村家庭的数倍。数据显示，“城市家庭在每个学生上的支出平均是农村家庭的2.5倍”（文件247040fre），在学前教育阶段甚至达到“三倍以上”（文件247040fre）。具体到小学，“城市家庭的教育支出是农村家庭的2.85倍”（文件247040fre，表3.17）。高额的支出能力是城市家庭经济优势的直接体现。
*   **贫困与失学高度相关**：最贫困的家庭主要集中在农村，而他们也正是失学风险最高的群体。“6至11岁最贫困的五分之一（Q1）家庭中，儿童失学风险接近44%，而最富裕的五分之一（Q5）家庭仅为13%”（文件247040fre）。此外，“72%的6至11岁失学儿童（EHSS）居住在农村”（文件247040fre），这强烈表明农村是经济弱势和受教育机会剥夺的重叠区域。

#### **2. 公私立学校差异：反映了家庭经济分层**
*   **私立学校家庭支出远高于公立学校**：选择私立教育意味着家庭需要承担高昂得多的费用。数据显示，在小学阶段，“家庭为就读私立学校的孩子平均支

### 4. 保存结构化数据

In [20]:
df.to_pickle(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Unstructured Data for LLM\Code process data\analysis_results.pkl')
df[['filename', 'relevant_description']].to_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Unstructured Data for LLM\Code process data\relevant_descriptions.csv', index=False, encoding='utf-8-sig')

with open(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Unstructured Data for LLM\Code process data\expert_report.txt', 'w', encoding='utf-8') as f:
    f.write(final_analysis)

# 二、 天气文献分析

### 1. Prompt

In [11]:
#任务一：提炼关键句；任务二：根据关键句回答问题
#任务一和任务二的prompt
EXTRACTION_INSTRUCTION = """
你的任务是从提供的文本块中，找出与以下话题相关的论述：
1.	科特迪瓦的气温、降雨、风速、气压等天气指标的平均情况、波动情况。
2.	ITCZ（赤道低压带/热带辐合带）、季风与哈马丹风（Harmattan）如何影响科特迪瓦的气候。
3.	科特迪瓦不同区域的气候差异。

请直接输出找到的论述原文（保持原文语言，不要翻译），严禁凭空臆造。
输出格式必须是一个 Python 列表格式，例如：
["原文论述1", "原文论述2", "原文论述3"]
如果没找到相关论述，请输出：[]。不要输出任何多余的解释文字。
"""

EXPERT_PROMPT = """
你是一个气候领域的资深专家。请深入分析以下来自科特迪瓦气候相关文件的论述，回答问题：
“请根据输入的材料分析科特迪瓦的气候特征，重点对比气温、降雨、风速、气压的年内波动。然后说明科特迪瓦作为热带雨林和热带草原气候交替的地区，哪个或哪些指标最适合作为划分季节和衡量气候变化的核心指标？”

要求：
1. 确保真实性与逻辑性。
2. 回答完后，必须明确列出你的各个观点分别是基于哪些文件的哪些论述得出的。在你得出的每个观点后添加括号说明，括号内的内容为此观点基于的文件名，和论述原文。
"""

### 2. 任务一：提炼各文件关键句

In [5]:
# 加载文件
folder_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Unstructured Data for LLM\txt about climate'

txt_files = glob.glob(os.path.join(folder_path, '*.txt'))
data = []
for file_path in txt_files:
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
        data.append({'filename': os.path.basename(file_path), 'text': content})

df = pd.DataFrame(data)
# 过滤非拉丁字符
# df = df[df['text'].apply(is_latin_alphabet)]

# 分块
df['text_chunks'] = df['text'].apply(lambda x: split_text_into_chunks(x))
df['relevant_description'] = [[] for _ in range(len(df))]


# --- 优化后的执行主流程 ---
print(f"开始处理，共计 {len(df)} 个文件...")
for index, row in df.iterrows():
    all_findings = []
    chunks = row['text_chunks']
    print(f"\n正在处理文件 [{index+1}/{len(df)}]: {row['filename']} (共 {len(chunks)} 个分块)")
    
    # 为分块处理增加内部进度条
    for i, chunk in enumerate(chunks):
        #print(chunk)#######@@
        # 实时打印进度
        print(f"  -> 正在提取第 {i+1}/{len(chunks)} 块...", end='\r') 
        
        raw_result = call_llm(chunk, EXTRACTION_INSTRUCTION)
        print(f"LLM原文返回: {raw_result}")############@@@@
        findings = parse_list_result(raw_result)
        all_findings.extend(findings)
        time.sleep(WAIT_TIME)
    
    df.at[index, 'relevant_description'] = list(set(all_findings))
    print(f"\n  Done! 该文件提取到 {len(all_findings)} 条论述。")

开始处理，共计 25 个文件...

正在处理文件 [1/25]: 010088055.txt (共 3 个分块)
LLM原文返回: []1/3 块...
LLM原文返回: ["Considering annual precipitation, the 1961-1980 period was wetter in the south along the coast and on the western side of the country, with cumulative rainfall ranging in the south from 1600 to 1700 mm/ year, with maxima in the extreme southwest and southeast, to 1300-1400 mm along the western side. The rest of the country, particularly the center and the north, had lower rainfall values ranging from 1000 to 1300 mm, with persistent minima in the center.", "The maps of the number of rainy days (RR1) showed the same pattern as PRCPTOT, with maxima ranging from 94 to 107 days in 1961-1980 over the western side and southern region of the country, respectively, while the other part had a weaker number of rainy days (80-90 days).", "The maps of rainfall intensity (SDII) showed a completely different pattern, with higher values over the south (14-18 mm/day in 1961-1980) and weaker values in the center an

### 3. 任务二：根据关键句回答问题

In [12]:
# 汇总所有非空的论述
summary_context = ""
for _, row in df.iterrows():
    if row['relevant_description']:
        summary_context += f"--- 文件: {row['filename']} ---\n"
        summary_context += "\n".join(row['relevant_description']) + "\n\n"


print("\n正在生成专家分析报告...")
final_analysis = call_llm(summary_context, EXPERT_PROMPT)
print("\n--- 任务二：专家分析结果 ---")
print(final_analysis)


正在生成专家分析报告...

--- 任务二：专家分析结果 ---
### 分析科特迪瓦的气候特征及季节划分核心指标

基于提供的多个气候相关文件，我对科特迪瓦的气候特征进行了深入分析，重点对比气温、降雨、风速和气压的年内波动。科特迪瓦位于西非，地理纬度介于约4°N至10°N之间，受热带气候主导，气候类型从南部的赤道气候过渡到北部的热带草原气候，并受西非季风、哈马丹风（harmattan）和热带辐合带（ITCZ）季节性迁移的强烈影响。以下分析基于文件内容，确保真实性和逻辑性。

#### 1. **气温的年内波动**
- **特征**：科特迪瓦的气温整体温暖，年均温约26-28°C，但存在明显的空间和季节差异。南部沿海地区气温较稳定，年内波动小（如阿比让年均温约26°C），而北部内陆地区气温波动较大，日温差可达10°C以上（如Korhogo日温范围约32°C）。气温年内波动表现为：最热月份通常在2-4月（北部可达40°C），最凉月份在7-8月（沿海约25°C）。长期趋势显示气温上升，如1960-2000年间平均温升高约1°C。
- **依据**：  
  - 文件"Climat de la Côte d'Ivoire — Wikipédia.txt"指出："Les températures moyennes mensuelles qui étaient de 26,5 °C dans les années 1960 à 1975, sont passées à 27,4 °C dès les années 2000, soit une hausse d’environ 1 °C en 25 ans" 和 "La Côte d'Ivoire connaît généralement des variations importantes de température entre le Nord et le Sud, mais également sur le long de l’année en fonction des saisons。"  
  - 文件"Ivory Coast climate_ average weather, temperature, rain, when to go.txt"描述："In the northern city of Korhog

### 4. 保存结构化数据

In [13]:
df.to_pickle(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Unstructured Data for LLM\Code process data\analysis_results_climate.pkl')
df[['filename', 'relevant_description']].to_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Unstructured Data for LLM\Code process data\relevant_descriptions_climate.csv', index=False, encoding='utf-8-sig')

with open(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Unstructured Data for LLM\Code process data\expert_report_climate.txt', 'w', encoding='utf-8') as f:
    f.write(final_analysis)